# Subflow - Group Steps into Containers

This notebook demonstrates how to use the `Subflow` class to group related steps into visual containers.

In [ ]:
import sys
print(sys.executable)
print(sys.path)

: 

## Example 1: Simple Subflow

Create a subflow container with steps inside it.

In [ ]:
from ipydagflow import Step, StepDAG, Subflow

# Create a subflow container
etl = Subflow(id="etl", label="ETL Pipeline", width=300, height=200)

# Create steps inside the subflow
extract = Step(id="extract", label="Extract", step_type="datasource")
transform = Step(id="transform", label="Transform", step_type="box")
load = Step(id="load", label="Load", step_type="datasource")

# Add steps to the subflow
etl.add_steps(extract, transform, load)

# Connect steps within the subflow
extract.add_child(transform)
transform.add_child(load)

# Render
dag = StepDAG()
dag.add_subflow(etl)
dag.render()

## Example 2: Multiple Subflows

Create multiple subflows and connect them.

In [ ]:
# Test: just two subflows, no children
from ipydagflow import DynamicDAG

nodes = [
    {"id": "sf1", "type": "group", "data": {"label": "Subflow 1"}, "style": {"width": 200, "height": 120}},
    {"id": "sf2", "type": "group", "data": {"label": "Subflow 2"}, "style": {"width": 200, "height": 120}},
]
edges = [
    {"id": "e1", "source": "sf1", "target": "sf2"}
]

DynamicDAG(nodes=nodes, edges=edges)

## Example 3: Subflow with External Steps

Connect a subflow to external steps.

In [ ]:
# External input step
input_data = Step(id="input", label="Input Data", step_type="datasource")

# Processing subflow
processing = Subflow(id="proc", label="Processing", width=280, height=150)
step1 = Step(id="step1", label="Step 1", step_type="box")
step2 = Step(id="step2", label="Step 2", step_type="box")
processing.add_steps(step1, step2)
step1.add_child(step2)

# External output step
output_data = Step(id="output", label="Output Data", step_type="datasource")

# Connect: input -> subflow -> output
input_data.add_child(processing)
processing.add_child(output_data)

# Render
dag = StepDAG()
dag.add_step(input_data)
dag.add_subflow(processing)
dag.add_step(output_data)
dag.render()

## Example 4: ML Pipeline with Subflows

A machine learning pipeline organized into logical groups.

In [ ]:
# Data preparation subflow
data_prep = Subflow(id="data_prep", label="Data Preparation", width=320, height=200)
load_data = Step(id="load", label="Load Data", step_type="datasource")
clean_data = Step(id="clean", label="Clean", step_type="box")
split_data = Step(id="split", label="Train/Test Split", step_type="box")
data_prep.add_steps(load_data, clean_data, split_data)
load_data.add_child(clean_data)
clean_data.add_child(split_data)

# Feature engineering subflow
feature_eng = Subflow(id="feature_eng", label="Feature Engineering", width=320, height=200)
feat_extract = Step(id="extract_feat", label="Extract Features", step_type="box")
feat_select = Step(id="select_feat", label="Select Features", step_type="box")
feat_scale = Step(id="scale_feat", label="Scale Features", step_type="box")
feature_eng.add_steps(feat_extract, feat_select, feat_scale)
feat_extract.add_child(feat_select)
feat_select.add_child(feat_scale)

# Model training subflow
model_train = Subflow(id="model_train", label="Model Training", width=320, height=200)
train = Step(id="train", label="Train Model", step_type="box")
evaluate = Step(id="evaluate", label="Evaluate", step_type="box")
save_model = Step(id="save", label="Save Model", step_type="datasource")
model_train.add_steps(train, evaluate, save_model)
train.add_child(evaluate)
evaluate.add_child(save_model)

# Connect subflows
data_prep.add_child(feature_eng)
feature_eng.add_child(model_train)

# Render
dag = StepDAG()
dag.add_subflows(data_prep, feature_eng, model_train)
dag.render()

## Example 5: Subflow with Custom Metadata

In [ ]:
# Subflow with metadata
batch_job = Subflow(
    id="batch",
    label="Batch Job",
    width=300,
    height=180,
    data={
        "schedule": "0 0 * * *",  # Daily at midnight
        "timeout": "2h",
        "retries": 3
    }
)

# Steps with metadata
read = Step(
    id="read",
    label="Read",
    step_type="datasource",
    data={"source": "s3://bucket/input"}
)
process = Step(
    id="process",
    label="Process",
    step_type="box",
    data={"memory": "8GB", "cpu": 4}
)
write = Step(
    id="write",
    label="Write",
    step_type="datasource",
    data={"destination": "s3://bucket/output"}
)

batch_job.add_steps(read, process, write)
read.add_child(process)
process.add_child(write)

dag = StepDAG()
dag.add_subflow(batch_job)
dag.render()

## API Summary

### Subflow Class
```python
subflow = Subflow(
    id="unique_id",           # Required: unique identifier
    label="Display Name",     # Required: display label
    width=200,                 # Optional: container width
    height=150,                # Optional: container height
    data={...}                 # Optional: additional metadata
)

# Methods
subflow.add_step(step)             # Add single step to container
subflow.add_steps(s1, s2, s3)      # Add multiple steps to container
subflow.remove_step(step)          # Remove step from container
subflow.add_child(other)           # Add DAG edge to another subflow/step
subflow.add_parent(other)          # Add DAG edge from another subflow/step
```

### StepDAG with Subflows
```python
dag = StepDAG()

# Methods
dag.add_subflow(subflow)           # Add single subflow
dag.add_subflows(sf1, sf2)         # Add multiple subflows
dag.get_subflow("subflow_id")      # Get subflow by ID
dag.get_all_subflows()             # Get all subflows
dag.render()                       # Render as DynamicDAG widget
```